In [5]:
import joblib

vectorizer = joblib.load("vectorizer.jb")
model = joblib.load("lr_model.jb")


## Label normalization

`scikit-learn` stores whatever class values the model was trained on in
`model.classes_`. If training data encoded labels differently than the raw
`1 == Real` assumption (e.g. `0 = Real / 1 = Fake`, or string/bool labels),
predictions would come out mislabeled. This function normalizes any of
those raw prediction formats into a consistent `"Real"` / `"Fake"` string.


In [6]:
def _normalize_label(raw_prediction) -> str:
    value = raw_prediction
    # numpy scalars/bools -> plain python
    if hasattr(value, "item"):
        value = value.item()

    if isinstance(value, bytes):
        value = value.decode(errors="ignore")

    if isinstance(value, str):
        v = value.strip().upper()
        if v in ("REAL", "TRUE", "1"):
            return "Real"
        if v in ("FAKE", "FALSE", "0"):
            return "Fake"
    
        return value.title()

    if isinstance(value, bool):
        return "Real" if value else "Fake"

    
    return "Real" if int(value) == 1 else "Fake"


## Classification

Runs the TF-IDF + Logistic Regression pipeline on any block of text and
returns `(label, confidence)`. This function is UI-agnostic — it has no
knowledge of Streamlit session state. Any manual "flip labels" override
is applied by the caller (`app.py`), not here, so this notebook stays pure
ML logic.


In [7]:
def classify_text(text: str):
    """Run the TF-IDF + Logistic Regression pipeline on any block of text
    and return (label, confidence). label is "Real" or "Fake"; confidence
    is a float in [0, 1] or None if the model has no predict_proba."""
    transformed = vectorizer.transform([text])
    raw_prediction = model.predict(transformed)[0]
    label = _normalize_label(raw_prediction)

    confidence = None
    if hasattr(model, "predict_proba"):
        try:
            proba = model.predict_proba(transformed)[0]
            confidence = float(max(proba))
        except Exception:
            confidence = None

    return label, confidence


In [8]:
def get_model_classes():
    """Expose the raw classes_ the model was trained on, mainly for the
    diagnostic caption shown in app.py's Advanced sidebar section."""
    return list(getattr(model, "classes_", []))
